In [ ]:
import json
import datetime
from dataclasses import replace
from pathlib import Path
 
from openai import OpenAI
 
from NegoLib.Entities.Agents     import BUDGET_MERCHANT, LOCAL_SUPPLIER
from NegoLib.Entities.scenarios  import KIOSK_COLA, OFFICE_LOBBY
from NegoLib.Entities.utility    import merchant_utility, supplier_utility
from NegoLib.prompt_ensemble import (
    build_merchant_prompt,
    build_supplier_prompt,
    build_judge_prompt,
    Agent_call,
    Judge_call,
)

from NegoLib.exp_run import batch_simple_exp, quick_save
 
 
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
 
client = OpenAI(
    api_key="sk-81e3e6eae33d4486ad4127e0cb3630a6",
    base_url="https://api.deepseek.com",
)
 
MODEL = "deepseek-reasoner"

merchant   = replace(BUDGET_MERCHANT, max_rounds=20)
supplier   = replace(LOCAL_SUPPLIER, max_rounds=20)
scenarios  = [KIOSK_COLA, KIOSK_COLA, OFFICE_LOBBY, OFFICE_LOBBY]
STRATEGY_M = "cooperative"
STRATEGY_S = "cooperative"


print(f"Running experiments with MODEL={MODEL}...")
experiment_output = batch_simple_exp(
    client = client,
    MODEL = MODEL,
    N_EXPS=10,
    Scenarios=scenarios,
    Merchants = [merchant],
    Suppliers=[supplier],
    Merchant_strategies=[STRATEGY_M],
    Supplier_strategies=[STRATEGY_S],
    verbose = False,
)


quick_save(experiment_output)


# 60 min


Running experiments with MODEL=deepseek-reasoner...

Saved to: quick_save_20260411_033106.json


In [7]:
# ---------------------------------------------------------------------------
# Load experiment_output from latest JSON in ./results
# ---------------------------------------------------------------------------

import json
from pathlib import Path

results_dir = Path("results")
if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found: {results_dir.resolve()}")

candidates = sorted(results_dir.glob("quick_save_*.json"), key=lambda p: p.stat().st_mtime)
if not candidates:
    candidates = sorted(results_dir.glob("*.json"), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError(f"No json files found under: {results_dir.resolve()}")

json_path = candidates[-1]
with open(json_path, "r", encoding="utf-8") as f:
    experiment_output = json.load(f)

if "runs" not in experiment_output:
    raise KeyError(f"Loaded json does not contain 'runs': {json_path}")

print(f"Loaded experiment_output from: {json_path}")
print(f"Number of runs: {len(experiment_output['runs'])}")


Loaded experiment_output from: results/exp1_20260411_033106.json
Number of runs: 10


In [9]:
# ---------------------------------------------------------------------------
# Price collection + n -> (n+1) supplier prediction
# Uses existing experiment_output from the initial cell (no changes to run logic).
# No train/test split: fit all points directly.
# Also exports point-vs-fitted results to file.
# ---------------------------------------------------------------------------

import csv
import re
import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


def _extract_price_map_from_answer(answer: str, product_names: list[str]) -> dict[str, float]:
    prices = {}
    for name in product_names:
        pattern = rf"{re.escape(name)}\s*:\s*\$?\s*([0-9][0-9,]*(?:\.[0-9]+)?)\s*per unit"
        m = re.search(pattern, answer, flags=re.IGNORECASE)
        if m:
            prices[name] = float(m.group(1).replace(",", ""))

    if len(prices) != len(product_names):
        raw = re.findall(r"\$\s*([0-9][0-9,]*(?:\.[0-9]+)?)\s*per unit", answer, flags=re.IGNORECASE)
        vals = [float(x.replace(",", "")) for x in raw]
        if len(vals) == len(product_names):
            prices = {name: vals[i] for i, name in enumerate(product_names)}
    return prices


def _mean_price(price_map: dict[str, float]) -> float | None:
    if not price_map:
        return None
    return float(sum(price_map.values()) / len(price_map))


def _rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def _r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else 1.0


def _feature_matrix(m: np.ndarray, s: np.ndarray, kind: str) -> np.ndarray:
    if kind == "linear":
        return np.column_stack([np.ones_like(m), m, s])
    if kind == "quadratic":
        return np.column_stack([np.ones_like(m), m, s, m * m, s * s, m * s])
    if kind == "cubic":
        return np.column_stack(
            [
                np.ones_like(m),
                m,
                s,
                m * m,
                s * s,
                m * s,
                m ** 3,
                s ** 3,
                (m * m) * s,
                m * (s * s),
            ]
        )
    raise ValueError(f"Unknown kind: {kind}")


def _fit_lstsq_all(m: np.ndarray, s: np.ndarray, y: np.ndarray, kind: str) -> dict:
    X = _feature_matrix(m, s, kind)
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ beta
    return {
        "model": kind,
        "rmse_all": _rmse(y, pred),
        "r2_all": _r2(y, pred),
        "n_params": int(len(beta)),
        "pred_all": pred,
    }


def _knn_predict(x_train: np.ndarray, y_train: np.ndarray, x_query: np.ndarray, k: int) -> np.ndarray:
    preds = []
    for q in x_query:
        d = np.sqrt(np.sum((x_train - q) ** 2, axis=1))
        idx = np.argsort(d)[:k]
        preds.append(float(np.mean(y_train[idx])))
    return np.array(preds, dtype=float)


# ---------------------------------------------------------------------------
# Build round-level rows and transition rows
# ---------------------------------------------------------------------------
round_rows = []
for run_id, run in enumerate(experiment_output["runs"], start=1):
    product_names = [o["product_name"] for o in run["scenario"]["orders"]]

    for r in run["rounds"]:
        m_answer = r["merchant"]["answer"]
        s_answer = r["supplier"]["answer"]

        m_prices = _extract_price_map_from_answer(m_answer, product_names)
        s_prices = _extract_price_map_from_answer(s_answer, product_names)

        round_rows.append(
            {
                "run_id": run_id,
                "round": int(r["round"]),
                "merchant_price_mean": _mean_price(m_prices),
                "supplier_price_mean": _mean_price(s_prices),
                "merchant_prices": m_prices,
                "supplier_prices": s_prices,
            }
        )

print(f"Collected {len(round_rows)} round rows from {len(experiment_output['runs'])} runs.")

transitions = []
by_run = {}
for row in round_rows:
    by_run.setdefault(row["run_id"], []).append(row)

for run_id, rows in by_run.items():
    rows = sorted(rows, key=lambda x: x["round"])
    for i in range(len(rows) - 1):
        cur = rows[i]
        nxt = rows[i + 1]

        m_n = cur["merchant_price_mean"]
        s_n = cur["supplier_price_mean"]
        s_np1 = nxt["supplier_price_mean"]

        if m_n is None or s_n is None or s_np1 is None:
            continue

        transitions.append(
            {
                "run_id": run_id,
                "round_n": cur["round"],
                "merchant_n": m_n,
                "supplier_n": s_n,
                "supplier_n_plus_1": s_np1,
            }
        )

print(f"Usable transitions: {len(transitions)}")
if not transitions:
    raise ValueError("No usable transitions found. Check output format or extraction regex.")


# ---------------------------------------------------------------------------
# Export base tables
# ---------------------------------------------------------------------------
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

if "json_path" in globals() and json_path is not None:
    source_prefix = Path(json_path).stem.split("_", 1)[0]
else:
    source_prefix = "experiment"

round_csv_path = output_dir / f"{source_prefix}_assump_check_round_rows.csv"
transitions_csv_path = output_dir / f"{source_prefix}_assump_check_transitions.csv"
excel_path = output_dir / f"{source_prefix}_assump_check_data.xlsx"
fit_csv_path = output_dir / f"{source_prefix}_assump_check_points_and_fits.csv"

round_export = []
for r in round_rows:
    round_export.append(
        {
            "run_id": r["run_id"],
            "round": r["round"],
            "merchant_price_mean": r["merchant_price_mean"],
            "supplier_price_mean": r["supplier_price_mean"],
            "merchant_prices_json": json.dumps(r["merchant_prices"], ensure_ascii=False),
            "supplier_prices_json": json.dumps(r["supplier_prices"], ensure_ascii=False),
        }
    )

with open(round_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "run_id",
            "round",
            "merchant_price_mean",
            "supplier_price_mean",
            "merchant_prices_json",
            "supplier_prices_json",
        ],
    )
    writer.writeheader()
    writer.writerows(round_export)

with open(transitions_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["run_id", "round_n", "merchant_n", "supplier_n", "supplier_n_plus_1"],
    )
    writer.writeheader()
    writer.writerows(transitions)


# ---------------------------------------------------------------------------
# Fit on all points (no split)
# ---------------------------------------------------------------------------
m = np.array([t["merchant_n"] for t in transitions], dtype=float)
s = np.array([t["supplier_n"] for t in transitions], dtype=float)
y = np.array([t["supplier_n_plus_1"] for t in transitions], dtype=float)

model_runs = []
model_runs.append(_fit_lstsq_all(m, s, y, "linear"))
model_runs.append(_fit_lstsq_all(m, s, y, "quadratic"))
model_runs.append(_fit_lstsq_all(m, s, y, "cubic"))

X_knn = np.column_stack([m, s])
mu = np.mean(X_knn, axis=0)
sigma = np.std(X_knn, axis=0)
sigma[sigma == 0] = 1.0
X_knn_z = (X_knn - mu) / sigma

k = min(7, len(X_knn_z))
pred_knn = _knn_predict(X_knn_z, y, X_knn_z, k)
model_runs.append(
    {
        "model": f"knn_k{k}",
        "rmse_all": _rmse(y, pred_knn),
        "r2_all": _r2(y, pred_knn),
        "n_params": 0,
        "pred_all": pred_knn,
    }
)

results = sorted(
    [
        {
            "model": r["model"],
            "rmse_all": r["rmse_all"],
            "r2_all": r["r2_all"],
            "n_params": r["n_params"],
        }
        for r in model_runs
    ],
    key=lambda r: r["rmse_all"],
)

print("\nAll-point fit results (sorted by RMSE):")
print(json.dumps(results, indent=2))


# ---------------------------------------------------------------------------
# Export point-vs-fitted table (the file you asked for)
# ---------------------------------------------------------------------------
pred_by_model = {r["model"]: np.array(r["pred_all"], dtype=float) for r in model_runs}

fit_rows = []
for i, t in enumerate(transitions):
    row = {
        "run_id": t["run_id"],
        "round_n": t["round_n"],
        "merchant_n": t["merchant_n"],
        "supplier_n": t["supplier_n"],
        "supplier_n_plus_1_true": t["supplier_n_plus_1"],
    }
    for model_name, pred in pred_by_model.items():
        row[f"pred_{model_name}"] = float(pred[i])
    fit_rows.append(row)

fieldnames = list(fit_rows[0].keys())
with open(fit_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(fit_rows)


# Optional Excel with all tables
excel_saved = False
if pd is not None:
    try:
        with pd.ExcelWriter(excel_path) as writer:
            pd.DataFrame(round_export).to_excel(writer, sheet_name="round_rows", index=False)
            pd.DataFrame(transitions).to_excel(writer, sheet_name="transitions", index=False)
            pd.DataFrame(fit_rows).to_excel(writer, sheet_name="points_and_fits", index=False)
        excel_saved = True
    except Exception as e:
        print(f"Excel export skipped ({type(e).__name__}: {e})")
else:
    print("Excel export skipped (pandas not available).")

print(f"Saved CSV: {round_csv_path}")
print(f"Saved CSV: {transitions_csv_path}")
print(f"Saved CSV: {fit_csv_path}")
if excel_saved:
    print(f"Saved Excel: {excel_path}")


# ---------------------------------------------------------------------------
# Plots (all points)
# ---------------------------------------------------------------------------
fit_curve_plot_path = output_dir / f"{source_prefix}_assump_check_points_and_fits.png"
scatter_plot_path = output_dir / f"{source_prefix}_assump_check_true_vs_pred.png"
rmse_bar_path = output_dir / f"{source_prefix}_assump_check_rmse_all.png"

plots_saved = {
    "points_and_fits_plot": None,
    "true_vs_pred_plot": None,
    "rmse_bar_plot": None,
}

if plt is not None:
    # Sort for cleaner line plot
    order = np.argsort(y)
    y_sorted = y[order]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(np.arange(len(y_sorted)), y_sorted, marker="o", linewidth=1.5, label="true")
    for r in model_runs:
        pred_sorted = np.array(r["pred_all"], dtype=float)[order]
        ax.plot(np.arange(len(pred_sorted)), pred_sorted, linewidth=1.2, label=r["model"])
    ax.set_xlabel("sample index (sorted by true value)")
    ax.set_ylabel("supplier_(n+1)")
    ax.set_title("All Points: True vs Fitted")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fit_curve_plot_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    plots_saved["points_and_fits_plot"] = str(fit_curve_plot_path)

    n_models = len(model_runs)
    ncols = 2
    nrows = int(np.ceil(n_models / ncols))
    fig2, axes = plt.subplots(nrows, ncols, figsize=(11, 4.5 * nrows))
    axes = np.array(axes).reshape(-1)
    y_min = float(np.min(y))
    y_max = float(np.max(y))
    if y_max <= y_min:
        y_max = y_min + 1e-6
    for i, r in enumerate(model_runs):
        ax = axes[i]
        pred = np.array(r["pred_all"], dtype=float)
        ax.scatter(y, pred, alpha=0.8)
        ax.plot([y_min, y_max], [y_min, y_max], "k--", linewidth=1)
        ax.set_title(f"{r['model']} | RMSE={r['rmse_all']:.4f}")
        ax.set_xlabel("True supplier_(n+1)")
        ax.set_ylabel("Fitted supplier_(n+1)")
    for j in range(n_models, len(axes)):
        axes[j].axis("off")
    fig2.tight_layout()
    fig2.savefig(scatter_plot_path, dpi=180, bbox_inches="tight")
    plt.close(fig2)
    plots_saved["true_vs_pred_plot"] = str(scatter_plot_path)

    fig3, ax3 = plt.subplots(figsize=(8, 4.5))
    names = [r["model"] for r in results]
    vals = [r["rmse_all"] for r in results]
    ax3.bar(names, vals)
    ax3.set_ylabel("RMSE (all points)")
    ax3.set_title("All-Point Fit Comparison")
    ax3.tick_params(axis="x", rotation=20)
    fig3.tight_layout()
    fig3.savefig(rmse_bar_path, dpi=180, bbox_inches="tight")
    plt.close(fig3)
    plots_saved["rmse_bar_plot"] = str(rmse_bar_path)

    print(f"Saved plot: {fit_curve_plot_path}")
    print(f"Saved plot: {scatter_plot_path}")
    print(f"Saved plot: {rmse_bar_path}")
else:
    print("Plotting skipped (matplotlib unavailable).")


# ---------------------------------------------------------------------------
# Save analysis json
# ---------------------------------------------------------------------------
analysis_output = {
    "metadata": {
        "timestamp": datetime.datetime.now().isoformat(),
        "model": MODEL,
        "n_runs": len(experiment_output["runs"]),
        "n_transitions": len(transitions),
        "fit_mode": "all_points",
    },
    "export_paths": {
        "round_csv": str(round_csv_path),
        "transitions_csv": str(transitions_csv_path),
        "points_and_fits_csv": str(fit_csv_path),
        "excel": str(excel_path) if excel_saved else None,
        "points_and_fits_plot": plots_saved["points_and_fits_plot"],
        "true_vs_pred_plot": plots_saved["true_vs_pred_plot"],
        "rmse_bar_plot": plots_saved["rmse_bar_plot"],
    },
    "model_results": results,
    "transitions": transitions,
    "round_rows": round_rows,
}

quick_save(analysis_output, save_path=output_dir / f"{source_prefix}_assump_check_from_experiment.json")





Collected 41 round rows from 10 runs.
Usable transitions: 24

All-point fit results (sorted by RMSE):
[
  {
    "model": "cubic",
    "rmse_all": 0.019974356661099885,
    "r2_all": 0.9846788605508334,
    "n_params": 10
  },
  {
    "model": "quadratic",
    "rmse_all": 0.022684920389927855,
    "r2_all": 0.9802384964948374,
    "n_params": 6
  },
  {
    "model": "linear",
    "rmse_all": 0.023715042736829714,
    "r2_all": 0.9784030069765272,
    "n_params": 3
  },
  {
    "model": "knn_k7",
    "rmse_all": 0.05143189507130889,
    "r2_all": 0.8984194597678756,
    "n_params": 0
  }
]
Excel export skipped (ModuleNotFoundError: No module named 'openpyxl')
Saved CSV: results/exp1_assump_check_round_rows.csv
Saved CSV: results/exp1_assump_check_transitions.csv
Saved CSV: results/exp1_assump_check_points_and_fits.csv
Saved plot: results/exp1_assump_check_points_and_fits.png
Saved plot: results/exp1_assump_check_true_vs_pred.png
Saved plot: results/exp1_assump_check_rmse_all.png


NameError: name 'datetime' is not defined